# Phase 3: 特征工程进行筛选工作

## 概述 Overview

本notebook主要是进行一些筛选工作

**超参数配置：**
- Logistic Regression: L2正则化, balanced权重
- Random Forest: 100棵树, balanced权重
- XGBoost: GPU加速, max_depth=6, balanced权重

**数据配置：**
- 半衰期阈值：SIF：270，SGF：250
- 半衰期异常值：700

**输入**: 
- feature-engineering文件夹替换了原本的mixture，使用区分度为90，数据集划分更加合理
- 在phase中使用特征工程处理，新增ChemBerta分子表征以后进行检查，主要检查其预测结果变化情况，这次可以只存一个了


---

## 1. 环境检查与导入 Environment Setup

In [6]:
# 环境检查
import sys
from pathlib import Path

# 添加项目根目录到路径
project_root = Path.cwd().parent
print(project_root)
sys.path.insert(0, str(project_root / "src"))

# 核心库导入
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import warnings
import json
warnings.filterwarnings('ignore')

# 机器学习
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)
from xgboost import XGBClassifier

# 检查GPU可用性
try:
    import torch
    gpu_available = torch.cuda.is_available()
    if gpu_available:
        print(f"✓ GPU可用: {torch.cuda.get_device_name(0)}")
    else:
        print("⚠ GPU不可用，将使用CPU训练")
except ImportError:
    gpu_available = False
    print("⚠ PyTorch未安装，将使用CPU训练")

# 设置显示选项
pd.set_option('display.max_columns', None)
plt.rcParams['figure.figsize'] = (12, 6)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ 所有库已成功导入")
print(f"✓ 项目根目录: {project_root}")

d:\RA\feature_extraction
⚠ GPU不可用，将使用CPU训练
✓ 所有库已成功导入
✓ 项目根目录: d:\RA\feature_extraction


In [7]:

def convert_numpy_types(obj):
    """递归转换numpy类型为Python原生类型"""
    import numpy as np
    if isinstance(obj, dict):
        return {k: convert_numpy_types(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_numpy_types(v) for v in obj]
    elif isinstance(obj, (np.integer, np.int64, np.int32)):
        return int(obj)
    elif isinstance(obj, (np.floating, np.float64, np.float32)):
        return float(obj)
    elif isinstance(obj, (np.bool_, bool)):
        return bool(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    else:
        return obj


## 2. 参数配置区 Configuration

**⚙️ 根据您的需求修改以下参数**

In [8]:
# ============== 参数配置区 ==============

CONFIG = {
    # 输入输出路径
    'processed_dir': project_root / 'data' / 'feature-engineering' / 'csv',
    'features_dir': project_root / 'data' / 'feature-engineering' / 'features',
    'cv_results_dir': project_root / 'data' / 'feature-engineering'/ 'outputs' / 'phase3' / 'cv_results',
    'feature_importance_dir': project_root / 'data' / 'feature-engineering' / 'outputs' / 'phase3' / 'feature_importance',
    'transfer_results_dir': project_root / 'data' / 'feature-engineering' / 'outputs' / 'phase3' / 'transfer_results',
    'figures_dir': project_root / 'data' / 'feature-engineering'  / 'figures' / 'phase3',
    'independent_results_dir': project_root / 'data' / 'feature-engineering' / 'outputs' / 'phase3' / 'independent_results',
    # 一个临时输出的文件夹
    'temp':project_root/'data'/'feature-engineering'/'outputs'/'temp',

    # 模型选择（可选: 'lr', 'rf', 'xgb'）
    'models_to_train': ['lr', 'rf', 'xgb'],
    
    # 交叉验证参数
    'n_folds': 5,
    'random_state': 42,
    
    # XGBoost参数
    'use_gpu': gpu_available,
    'xgb_max_depth': 6,
    'xgb_learning_rate': 0.1,
    'xgb_n_estimators': 100,
    
    # Random Forest参数
    'rf_n_estimators': 100,
    'rf_n_jobs': -1,
    
    # Logistic Regression参数
    'lr_max_iter': 1000,
    
    # 可视化参数
    'dpi': 300,
    'format': 'png',
    'display_plots': True,
    'max_display_plots': 8,

    # 阈值
    'threshold':{
        'SIF':270,
        'SGF':250
    },

    # 是否只提取单体分子
    'is_monomer': True,

    # 用于验证模型效果的数据名称
    'dataset_names':{
        'train':"feature-engineering_train_Morgan(1024)_Avalon(512)_ChemBERTa(384)",
        'test':"feature-engineering_test_Morgan(1024)_Avalon(512)_ChemBERTa(384)"
    },
    # message,标注信息
    'message':'Morgan(1024)_Avalon(512)_ChemBERTa(384)'
}

# 创建输出目录
for key in ['cv_results_dir', 'feature_importance_dir', 'transfer_results_dir', 'figures_dir', 'independent_results_dir','temp']:
    CONFIG[key].mkdir(parents=True, exist_ok=True)

print("配置参数:")
print(f"  模型: {CONFIG['models_to_train']}")
print(f"  交叉验证折数: {CONFIG['n_folds']}")
print(f"  GPU加速: {CONFIG['use_gpu']}")
print(f"  XGBoost参数: max_depth={CONFIG['xgb_max_depth']}, lr={CONFIG['xgb_learning_rate']}")

配置参数:
  模型: ['lr', 'rf', 'xgb']
  交叉验证折数: 5
  GPU加速: False
  XGBoost参数: max_depth=6, lr=0.1


## 3. 数据加载与二值化 Data Loading & Binarization

In [9]:
def load_and_binarize_dataset(npz_path: Path, csv_path: Path, target: str, is_monomer: bool = None):
    """
    加载数据并将标签二值化，同时可选择只保留monomer或非monomer样本
    (2025-12-15新增功能)
    
    Args:
        npz_path: NPZ特征文件
        csv_path: 处理后的CSV文件（包含分钟标签）
        target: 'SIF' or 'SGF'
        is_monomer: 如果为True，只保留monomer；False，只保留非monomer；None不筛选
    
    Returns:
        X, y_binary, median_threshold, feature_names
    """
    # 加载NPZ特征
    data = np.load(npz_path, allow_pickle=True)
    X = data['X']
    feature_names = data['feature_names']
    ids_npz = data['ids']
    
    # 加载CSV获取分钟标签
    df = pd.read_csv(csv_path)
    df['id'] = df['id'].astype(str)
    
    # ID匹配：这里是建立一个映射数值
    id_to_idx = {str(id_): idx for idx, id_ in enumerate(ids_npz)}
    valid_indices = [] # 确定当前任务用哪一行
    valid_labels = []
    
    label_col = f"{target}_minutes" # 根据任务目标获取分钟数目
    for _, row in df.iterrows():
        row_id = str(row['id'])
        if row_id in id_to_idx:
            #===================================过滤无效标签=========================================
            label= row[label_col]
            if label == -1 or pd.isna(label):# 判断标签有效（对应的任务就只能做对应的二值化）
                continue
            if is_monomer is not None and row['is_monomer'] != is_monomer:  # 判断是否满足monomer条件
                continue
            if row[f"{target}_minutes"]>700: #特殊情况处理，如果是SIF_minuters>700
                continue   
            #=======================================================================================
            # 符合条件则加入
            valid_indices.append(id_to_idx[row_id]) # 获取对应数据的索引/序号
            valid_labels.append(label)              # 获取label
    
    # 筛选有效样本
    X_valid = X[valid_indices]   # 根据序号直接获取到值，这里是numpy的基本用法之一，比如传入【3，7】那么得到的就是第三行和第七行的数值
    y_minutes = np.array(valid_labels)  # 获取y值

    
    
    # 设定阈值========================
    median =None
    if target=='SIF':
        median = CONFIG['threshold']['SIF']
    elif target=='SGF':
        median = CONFIG['threshold']['SGF']

    # 根据阈值判断是否稳定
    y_binary = (y_minutes >= median).astype(int)  # 1=稳定, 0=不稳定
    
    print(f"  样本数: {len(X_valid)}")
    print(f"  中位数阈值（根据数值分析得到的结果）: {median:.1f} 分钟")
    print(f"  稳定/不稳定: {np.sum(y_binary==1)}/{np.sum(y_binary==0)}")
    
    return X_valid, y_binary, median, feature_names


# 加载所有数据集
datasets_data = {}
npz_files = [
    CONFIG['features_dir']/f"{CONFIG['dataset_names']['train']}.npz",
    CONFIG['features_dir']/f"{CONFIG['dataset_names']['test']}.npz"
]
# 选择加载数据的时候就指定monomer或非monomer样本
is_monomer = CONFIG['is_monomer'] # 仅加载monomer样本，设置为False则加载非monomer样本，None则不筛选

print(f"加载并二值化 {len(npz_files)} 个数据集:\n")
for npz_file in npz_files:
    dataset_name = npz_file.stem.replace('', '')# 这行代码看起来没啥用，其实是历史遗留问题，不用管也不要动
    csv_file = CONFIG['processed_dir'] / f"{dataset_name}.csv"
    
    print(f"{dataset_name}:")

    # 此处新增数据集加载功能（第四个参数）
    
    # SIF
    X_sif, y_sif, median_sif, feat_names = load_and_binarize_dataset(npz_file, csv_file, 'SIF',is_monomer)
    print(f"  SIF数据选择完成，只获得单体数据")
    
    # SGF
    X_sgf, y_sgf, median_sgf, _ = load_and_binarize_dataset(npz_file, csv_file, 'SGF',is_monomer)
    print(f"  SGF数据选择完成，只获取单体数据\n")
    
    datasets_data[dataset_name] = {
        'X_sif': X_sif,
        'y_sif': y_sif,
        'median_sif': median_sif,
        'X_sgf': X_sgf,
        'y_sgf': y_sgf,
        'median_sgf': median_sgf,
        'feature_names': feat_names,
    }

print(f"✓ 数据加载完成！共 {len(datasets_data)} 个数据集")

#=============对数据进行一些处理=========================
# 字典序
print("查看数据集类型",type(datasets_data))   
# dict_keys(['sif_sgf_second', 'US20140294902A1', 'US9624268', 'US9809623B2', 'WO2017011820A2'])
print("尝试输出数据查看情况",datasets_data.keys()) 


#=============上方为数据的处理区域=======================

加载并二值化 2 个数据集:

feature-engineering_train_Morgan(1024)_Avalon(512)_ChemBERTa(384):
  样本数: 391
  中位数阈值（根据数值分析得到的结果）: 270.0 分钟
  稳定/不稳定: 145/246
  SIF数据选择完成，只获得单体数据
  样本数: 296
  中位数阈值（根据数值分析得到的结果）: 250.0 分钟
  稳定/不稳定: 98/198
  SGF数据选择完成，只获取单体数据

feature-engineering_test_Morgan(1024)_Avalon(512)_ChemBERTa(384):
  样本数: 123
  中位数阈值（根据数值分析得到的结果）: 270.0 分钟
  稳定/不稳定: 31/92
  SIF数据选择完成，只获得单体数据
  样本数: 107
  中位数阈值（根据数值分析得到的结果）: 250.0 分钟
  稳定/不稳定: 25/82
  SGF数据选择完成，只获取单体数据

✓ 数据加载完成！共 2 个数据集
查看数据集类型 <class 'dict'>
尝试输出数据查看情况 dict_keys(['feature-engineering_train_Morgan(1024)_Avalon(512)_ChemBERTa(384)', 'feature-engineering_test_Morgan(1024)_Avalon(512)_ChemBERTa(384)'])


## 3.1 permutation检查运行结果

输出的应该是特征打分的情况

In [ ]:
# 
import random
import numpy as np
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import json
import random
# 选择模型：这里已经强制了随机种子1-100000（获取随机种子，保证每次的运行结果都有点区别，别的不好说）
def get_model(model_name: str, use_gpu: bool = False):
    
    CONFIG['random_state']=random.randint(1, 1000000)
    # print("选择模型为", model_name, "随机种子", CONFIG['random_state'])
    """
    创建模型实例
    """
    if model_name == 'lr':
        return LogisticRegression(
            max_iter=CONFIG['lr_max_iter'],
            class_weight='balanced',
            random_state=random.randint(1, 1000000)
        )
    elif model_name == 'rf':
        return RandomForestClassifier(
            n_estimators=CONFIG['rf_n_estimators'],
            class_weight='balanced',
            n_jobs=CONFIG['rf_n_jobs'],
            random_state=random.randint(1, 1000000)
        )
    elif model_name == 'xgb':
        params = {
            'max_depth': CONFIG['xgb_max_depth'],
            'learning_rate': CONFIG['xgb_learning_rate'],
            'n_estimators': CONFIG['xgb_n_estimators'],
            'random_state': random.randint(1, 1000000),
            'tree_method': 'hist',
        }
        if use_gpu:
            params['device'] = 'cuda:0'
        return XGBClassifier(**params)
    else:
        raise ValueError(f"Unknown model: {model_name}")

# 特征重要性分析函数
def calculate_feature_importance(model, X_test, y_test, feature_names, n_repeats=5):
    """
    计算排列重要性并返回特征重要性字典
    """
    try:
        # 计算排列重要性
        result = permutation_importance(
            model, X_test, y_test, 
            n_repeats=n_repeats, 
            random_state=42, # 这个是在控制shuffle的顺序，不过应该是不用管。。大概吧
            scoring='f1'  # 使用f1分数作为评估指标
        )
        
        # 创建特征重要性字典
        importance_dict = {}
        for i, feature_name in enumerate(feature_names):
            importance_dict[feature_name] = {
                'importance_mean': result.importances_mean[i],
                'importance_std': result.importances_std[i],
                'importance_values': result.importances[i].tolist()
            }
        
        return importance_dict
    
    except Exception as e:
        print(f"计算特征重要性时出错: {e}")
        return None

def independent_validate(X_train, y_train, X_test, y_test, model_name: str, 
                         dataset_train_name: str, 
                         dataset_test_name: str,
                         target: str):
    """
    在训练集上进行全量训练，在独立的测试集上进行单次验证，并计算特征重要性
    """
    # 检查类别情况，防止只有单类别无法计算指标
    unique_classes = np.unique(y_test)
    if len(unique_classes) < 2:
        print(f"    ⚠ 测试集只有类别 {unique_classes}，无法进行二分类评估，跳过该任务")
        return None

    # --- 核心训练过程 ---
    # 1. 创建模型
    model = get_model(model_name, CONFIG['use_gpu'])
    
    # 2. 直接在全量 X_train 上训练，不再有 Fold 迭代
    model.fit(X_train, y_train) 
    
    # --- 预测过程 ---
    y_pred = model.predict(X_test)
    proba = model.predict_proba(X_test)
    
    if proba.shape[1] == 2:
        y_proba = proba[:, 1]
    else:
        y_proba = None

    # --- 计算指标 ---
    metrics = {
        'accuracy': [accuracy_score(y_test, y_pred)],
        'precision': [precision_score(y_test, y_pred, average='binary', zero_division=0)],
        'recall': [recall_score(y_test, y_pred, average='binary', zero_division=0)],
        'f1': [f1_score(y_test, y_pred, average='binary', zero_division=0)],
        'auc': []
    }
    
    if y_proba is not None and len(unique_classes) > 1:
        metrics['auc'].append(roc_auc_score(y_test, y_proba))
    else:
        metrics['auc'].append(np.nan)

    # --- 计算特征重要性 ---
    # 生成特征名称（如果没有的话）
    if hasattr(X_test, 'feature_names'):
        feature_names = X_test.mns.tolist()
    else:
        feature_names = [f'feature_{i}' for i in range(X_test.shape[1])]
    
    # 计算排列重要性
    feature_importance_dict = calculate_feature_importance(
        model, X_test, y_test, feature_names, n_repeats=10
    )

    # --- 汇总结果 ---
    results = {
        'dataset': f'{dataset_train_name}_{dataset_test_name}',
        'target': target,
        'model': model_name,
        'fold': 1,
        'metrics': metrics,
        'mean_metrics': {k: np.nanmean(v) for k, v in metrics.items()},
        'std_metrics': {k: 0.0 for k in metrics.keys()},
        'feature_importance': feature_importance_dict
    }
    
    return results

# 创建存储所有特征重要性的字典
feature_importance_storage = {}

#====进行独立验证=========================
print("\n开始独立验证并计算特征重要性...\n")
independent_results_all = []

# 这里指定训练集和测试集名称
train_dataset_name = CONFIG['dataset_names']['train']
test_dataset_name = CONFIG['dataset_names']['test']

# 获取数据
train_data = datasets_data.get(train_dataset_name)
test_data = datasets_data.get(test_dataset_name)

# 简单判断后进行训练
if train_data is not None and test_data is not None:
    for target in ['SIF', 'SGF']:
        X_train = train_data[f'X_{target.lower()}']
        y_train = train_data[f'y_{target.lower()}']
        X_test = test_data[f'X_{target.lower()}']
        y_test = test_data[f'y_{target.lower()}']
        
        if len(y_train) == 0 or len(y_test) == 0:
            print(f"  {target}: 训练集或测试集无有效样本，跳过")
            continue
        
        print(f"\n独立验证 {train_dataset_name} -> {test_dataset_name} ({target}):")
        
        for model_name in CONFIG['models_to_train']:
            print(f"    {model_name.upper()}...", end=" ")

            try:
                results = independent_validate(
                    X_train, y_train, X_test, y_test,
                    model_name,
                    train_dataset_name,
                    test_dataset_name,
                    target
                )
            except ValueError as e:
                print(f"只有一种类别，跳过该任务")
                continue
            
            if results is None:
                print("跳过 ✓")
                continue

            independent_results_all.append(results)

            # 存储特征重要性到全局字典
            key = f"{train_dataset_name}_{test_dataset_name}_{target}_{model_name}"
            feature_importance_storage[key] = results['feature_importance']

# 保存特征重要性结果到单独文件
if feature_importance_storage:
    importance_path = CONFIG['temp'] / "feature_importance_summary.json"
    with open(importance_path, 'w') as f:
        json.dump(convert_numpy_types(feature_importance_storage), f, indent=2)
    print(f"\n✓ 特征重要性结果已保存到: {importance_path}")

print(f"✓ 独立验证完成！共 {len(independent_results_all)} 个实验")
print(f"  特征重要性存储字典包含 {len(feature_importance_storage)} 种情况")

# 打印特征重要性摘要
print("\n=== 特征重要性摘要 ===")
for key, importance_dict in feature_importance_storage.items():
    if importance_dict:
        # 获取最重要的3个特征
        sorted_features = sorted(
            importance_dict.items(), 
            key=lambda x: x[1]['importance_mean'], 
            reverse=True)[:3]
        
        print(f"\n{key}:")
        for feature, importance in sorted_features:
            print(f"  {feature}: {importance['importance_mean']:.4f} ± {importance['importance_std']:.4f}")


开始独立验证并计算特征重要性...


独立验证 feature-engineering_train_Morgan(1024)_Avalon(512)_ChemBERTa(384) -> feature-engineering_test_Morgan(1024)_Avalon(512)_ChemBERTa(384) (SIF):
    LR...     RF...     XGB... 
独立验证 feature-engineering_train_Morgan(1024)_Avalon(512)_ChemBERTa(384) -> feature-engineering_test_Morgan(1024)_Avalon(512)_ChemBERTa(384) (SGF):
    LR...     RF...     XGB... 
✓ 特征重要性结果已保存到: d:\RA\feature_extraction\data\feature-engineering\outputs\temp\feature_importance_summary.json
✓ 独立验证完成！共 6 个实验
  特征重要性存储字典包含 6 种情况

=== 特征重要性摘要 ===

feature-engineering_train_Morgan(1024)_Avalon(512)_ChemBERTa(384)_feature-engineering_test_Morgan(1024)_Avalon(512)_ChemBERTa(384)_SIF_lr:
  feature_4: 0.2023 ± 0.0544
  feature_12: 0.2023 ± 0.0544
  feature_1: 0.1890 ± 0.0687

feature-engineering_train_Morgan(1024)_Avalon(512)_ChemBERTa(384)_feature-engineering_test_Morgan(1024)_Avalon(512)_ChemBERTa(384)_SIF_rf:
  feature_1584: 0.0114 ± 0.0121
  feature_1737: 0.0114 ± 0.0121
  feature_1826: 0.0114 ± 0.